# Reranking structured data

## Overview
Traditional search methods often struggle with structured data formats like emails, database records, and tabular information. Cohere's rerank API can effectively handle these formats by converting them to YAML representations, enabling semantic ranking of complex data structures.

This tutorial demonstrates how to apply reranking to semi-structured and tabular data formats.

By the end, you will have implemented reranking for email data and employee records using YAML conversion techniques.

## What we'll cover
- Reranking semi-structured data
- Reranking tabular data


In [1]:
import cohere
import os
import yaml
import pandas as pd
from io import StringIO
import json
from dotenv import load_dotenv
load_dotenv(override=True)

top_n = 3
MODEL = os.environ["RERANK_MODEL"]
co = cohere.ClientV2(
    base_url=os.environ["RERANK_BASE_URL"],
    api_key=os.environ["RERANK_API_KEY"],
    timeout=300,
)

def return_results(results, documents):
    for idx, result in enumerate(results.results):
        print(f"Rank: {idx+1}")
        print(f"Score: {result.relevance_score}")
        print(f"Document: {documents[result.index]}\n")

## Reranking semi-structured data

In [2]:
query = "Any email about check ins?"

documents = [
    {
        "from": "hr@co1t.com",
        "to": "david@co1t.com",
        "date": "2024-06-24",
        "subject": "A Warm Welcome to Co1t!",
        "text": "We are delighted to welcome you to the team! As you embark on your journey with us, you'll find attached an agenda to guide you through your first week.",
    },
    {
        "from": "it@co1t.com",
        "to": "david@co1t.com",
        "date": "2024-06-24",
        "subject": "Setting Up Your IT Needs",
        "text": "Greetings! To ensure a seamless start, please refer to the attached comprehensive guide, which will assist you in setting up all your work accounts.",
    },
    {
        "from": "john@co1t.com",
        "to": "david@co1t.com",
        "date": "2024-06-24",
        "subject": "First Week Check-In",
        "text": "Hello! I hope you're settling in well. Let's connect briefly tomorrow to discuss how your first week has been going. Also, make sure to join us for a welcoming lunch this Thursday at noon—it's a great opportunity to get to know your colleagues!",
    },
]


yaml_docs = [yaml.dump(document, sort_keys=False) for document in documents]

results = co.rerank(
    model=MODEL, 
    query=query, 
    documents=yaml_docs,
    top_n=top_n
)

print(f"Query: {query}\n")
return_results(results, documents)

Query: Any email about check ins?

Rank: 1
Score: 0.9988734
Document: {'from': 'john@co1t.com', 'to': 'david@co1t.com', 'date': '2024-06-24', 'subject': 'First Week Check-In', 'text': "Hello! I hope you're settling in well. Let's connect briefly tomorrow to discuss how your first week has been going. Also, make sure to join us for a welcoming lunch this Thursday at noon—it's a great opportunity to get to know your colleagues!"}

Rank: 2
Score: 0.67542213
Document: {'from': 'hr@co1t.com', 'to': 'david@co1t.com', 'date': '2024-06-24', 'subject': 'A Warm Welcome to Co1t!', 'text': "We are delighted to welcome you to the team! As you embark on your journey with us, you'll find attached an agenda to guide you through your first week."}

Rank: 3
Score: 0.5521533
Document: {'from': 'it@co1t.com', 'to': 'david@co1t.com', 'date': '2024-06-24', 'subject': 'Setting Up Your IT Needs', 'text': 'Greetings! To ensure a seamless start, please refer to the attached comprehensive guide, which will assis

In [3]:
for doc in yaml_docs:
    print(doc)

from: hr@co1t.com
to: david@co1t.com
date: '2024-06-24'
subject: A Warm Welcome to Co1t!
text: We are delighted to welcome you to the team! As you embark on your journey with
  us, you'll find attached an agenda to guide you through your first week.

from: it@co1t.com
to: david@co1t.com
date: '2024-06-24'
subject: Setting Up Your IT Needs
text: Greetings! To ensure a seamless start, please refer to the attached comprehensive
  guide, which will assist you in setting up all your work accounts.

from: john@co1t.com
to: david@co1t.com
date: '2024-06-24'
subject: First Week Check-In
text: "Hello! I hope you're settling in well. Let's connect briefly tomorrow to discuss\
  \ how your first week has been going. Also, make sure to join us for a welcoming\
  \ lunch this Thursday at noon\u2014it's a great opportunity to get to know your\
  \ colleagues!"



## Reranking tabular data

In [4]:
# Create a demo CSV file
data = """name,role,join_date,email,status
Rebecca Lee,Senior Software Engineer,2024-07-01,rebecca@co1t.com,Full-time
Emma Williams,Product Designer,2024-06-15,emma@co1t.com,Full-time
Michael Jones,Marketing Manager,2024-05-20,michael@co1t.com,Full-time
Amelia Thompson,Sales Representative,2024-05-20,amelia@co1t.com,Part-time
Ethan Davis,Product Designer,2024-05-25,ethan@co1t.com,Contractor"""
data_csv = StringIO(data)

# Load the CSV file
df = pd.read_csv(data_csv)
df.head()


,name,role,join_date,email,status
0,Rebecca Lee,Senior Software Engineer,2024-07-01,rebecca@co1t.com,Full-time
1,Emma Williams,Product Designer,2024-06-15,emma@co1t.com,Full-time
2,Michael Jones,Marketing Manager,2024-05-20,michael@co1t.com,Full-time
3,Amelia Thompson,Sales Representative,2024-05-20,amelia@co1t.com,Part-time
4,Ethan Davis,Product Designer,2024-05-25,ethan@co1t.com,Contractor


In [5]:
# Define the documents
employees = df.to_dict("records")

# Convert the documents to YAML format
yaml_docs = [yaml.dump(doc, sort_keys=False) for doc in employees]

# Add the user query
query = "Any full-time product designers who joined recently?"

# Rerank the documents
results = co.rerank(
    model=MODEL,
    query=query,
    documents=yaml_docs,
    top_n=top_n,
)

print(f"Query: {query}\n")
return_results(results, employees)


Query: Any full-time product designers who joined recently?

Rank: 1
Score: 0.9889107
Document: {'name': 'Emma Williams', 'role': 'Product Designer', 'join_date': '2024-06-15', 'email': 'emma@co1t.com', 'status': 'Full-time'}

Rank: 2
Score: 0.9317363
Document: {'name': 'Ethan Davis', 'role': 'Product Designer', 'join_date': '2024-05-25', 'email': 'ethan@co1t.com', 'status': 'Contractor'}

Rank: 3
Score: 0.68897086
Document: {'name': 'Michael Jones', 'role': 'Marketing Manager', 'join_date': '2024-05-20', 'email': 'michael@co1t.com', 'status': 'Full-time'}



In [6]:
for doc in yaml_docs:
    print(doc)

name: Rebecca Lee
role: Senior Software Engineer
join_date: '2024-07-01'
email: rebecca@co1t.com
status: Full-time

name: Emma Williams
role: Product Designer
join_date: '2024-06-15'
email: emma@co1t.com
status: Full-time

name: Michael Jones
role: Marketing Manager
join_date: '2024-05-20'
email: michael@co1t.com
status: Full-time

name: Amelia Thompson
role: Sales Representative
join_date: '2024-05-20'
email: amelia@co1t.com
status: Part-time

name: Ethan Davis
role: Product Designer
join_date: '2024-05-25'
email: ethan@co1t.com
status: Contractor



## Conclusion
This tutorial covered the application of Cohere's rerank API to structured data formats, showing how YAML conversion enables effective semantic ranking of emails and employee records.